In [132]:
import requests
import pandas as pd
from io import BytesIO

In [133]:
respuesta = requests.get(
    "https://www.datosabiertos.gob.pe/node/23482/download",
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
respuesta.raise_for_status()

df = pd.read_csv(BytesIO(respuesta.content))

/tmp/ipykernel_1075/3458993878.py:8: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(respuesta.content))


In [134]:
tabla = pd.DataFrame({"Nombre de variable": df.columns,
                      "Valores nulos": df.isnull().sum().values,
                      "Valores únicos": df.nunique().values,
                      "Tipo de variable": df.dtypes.astype(str).values})
print(tabla.to_string(index=False))

Nombre de variable  Valores nulos  Valores únicos Tipo de variable
      departamento              0              23           object
         provincia              0             124           object
          distrito              0             625           object
         localidad         123635           10380           object
        enfermedad              0               3           object
               ano              0              26            int64
            semana              0              53            int64
        diagnostic              0               3           object
           tipo_dx              0               2           object
            diresa             23              32          float64
            ubigeo              0             662            int64
          localcod         149652           13888           object
              edad              0             123            int64
         tipo_edad              0               3           ob

In [135]:
relevante = ["provincia", "distrito",                    # Valores relevantes seleccionados
             "ano", "semana", "diagnostic", "tipo_dx"]

df_piura = df[relevante].copy()[df["departamento"] == "PIURA"]
df_piura.head()

tabla = pd.DataFrame({
    "Nombre de variable": df_piura.columns,
    "Valores nulos": df_piura.isnull().sum().values,
    "Valores únicos": df_piura.nunique().values,
    "Tipo de variable": df_piura.dtypes.astype(str).values,})

print(tabla.to_string(index=False))

Nombre de variable  Valores nulos  Valores únicos Tipo de variable
         provincia              0               8           object
          distrito              0              61           object
               ano              0              26            int64
            semana              0              53            int64
        diagnostic              0               3           object
           tipo_dx              0               2           object


In [141]:
# ['PIURA', 'SULLANA', 'TALARA', 'AYABACA', 'MORROPON', 'HUANCABAMBA', 'SECHURA', 'PAITA']
df_piura_piura = df_piura[df_piura["provincia"] == "PIURA"]
df_piura_piura.sort_values(by="ano", ascending = True)
start_year = df_piura_piura["ano"].unique()[0]
end_year = df_piura_piura["ano"].unique()[-1]

ano, semana, empty = [], [], []
for y in range(start_year, end_year + 1):
  for w in range(1, 54):
    ano.append(y)
    semana.append(w)
    empty.append(0)
temporal = pd.DataFrame({"ano": ano, "semana": semana,
                         "A97.0P": empty, "A97.1P": empty, "A97.2P": empty,
                         "A97.0C": empty, "A97.1C": empty, "A97.2C": empty,})

for i in range(len(df_piura_piura)):
  temporal.loc[(temporal["ano"] == df_piura_piura.iloc[i]["ano"]) &
               (temporal["semana"] == df_piura_piura.iloc[i]["semana"]),
               df_piura_piura.iloc[i]["diagnostic"] + df_piura_piura.iloc[i]["tipo_dx"]] += 1

#
"""
print(var["ano"], var["semana"], var["diagnostic"], var["tipo_dx"])
print(var["diagnostic"] + var["tipo_dx"])
"""


'\nprint(var["ano"], var["semana"], var["diagnostic"], var["tipo_dx"])\nprint(var["diagnostic"] + var["tipo_dx"])\n'

In [142]:
df_piura["provincia"].unique()

array(['PIURA', 'SULLANA', 'TALARA', 'AYABACA', 'MORROPON', 'HUANCABAMBA',
       'SECHURA', 'PAITA'], dtype=object)

In [143]:
temporal.to_csv("df_piura_procesado.csv", index=False)